# Training the visual search model, start to finish

This notebook builds, explains, and trains the model that predicts
**where the eyes go first** in visual search — fitted to 114,232
first eye movements from 333 people across 11 published experiments.

**Scope:** the first saccade of each trial, launched from the screen
center. Later saccades need extra machinery (inhibition of return,
moving vantage points) and are out of scope.

**In and out:** a picture of the display, the goal (color + shape),
and the previous trials go in; a probability for every item comes
out. One predicted saccade is one random draw.

**Before running** (needs the public data from https://osf.io/q27ph/):

    python pool_data.py --data_dir ".../Data Files" --out_dir dataset
    python build_contexts.py

Needs numpy, pandas, matplotlib, torch. Training: ~10 minutes.


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from front_end import render, shape_for, item_positions, IMG
from build_contexts import (opponency_contrast, shape_match_map,
                            template_axis, sensory_salience_map)


## 1. Model structure

Three steps, MAP-first: everything is written into one 2-D map;
items only appear at the readout.

**Step 1 — the priority map.** Over pixel locations $x$:

$$ M(x) = \underbrace{\alpha_P\, P(x)}_{\text{sensory field}} + \underbrace{g_C\, C_T(x) + g_F\, S_T(x)}_{\text{goal field}} + \underbrace{\sum_{j}\big(\beta_T\, h_{T,j}+\beta_D\, h_{D,j}\big)\, G(x-x_j)}_{\text{selection history}} $$

- $P$: goal-independent sensory evidence: bottom-up color salience (§3). Its gain $\alpha_P$ is the explicit sensory-field gain.
- $C_T$: target-color evidence (§4), already signed so target-colored items are positive and opposite-colored items are negative. The one gain $g_C$ up-weights the target color and down-weights the distractor color together.
- $S_T$: target-shape evidence (§4); $g_F$ up-weights the target shape.
- $h_{T,j}, h_{D,j}$: item $j$'s target/distractor location
  histories (§5), painted at the item positions by a fixed kernel
  $G$ ($\sigma = 0.03$, peak height 1 — a stated assumption).
- Units: $P$, $C_T$, and $S_T$ are GREYSCALE — full scale is 1 ($C_T$ and $S_T$ pixels in $[-1,1]$, while background-transparent $P$ is in $[0,1]$), like $G$'s peak height 1.
  So every learned weight reads the same way: the priority
  delivered by a full-strength unit of its channel.

**Step 2 — point-sensing readout** (§7). One number per item:
the model SENSES the map at each item's center, like a range
sensor reading one value per direction:

$$ F_i = M(x_i). $$

**Step 3 — choice (softmax).** Exponentiate the six priorities and
divide by their sum. Higher priority means higher probability, but
never all of it — that softness is what captures the variability
in real eye movements:

$$ \Pr(\text{first saccade} = i) =
\frac{e^{F_i}}{\sum_j e^{F_j}}. $$

**Between trials** the histories are LEAKY ACCUMULATORS: every
location fades by $(1-\eta)$, and the location the item actually
occupied gets a boost of $\eta$. So $\eta$ is the WEIGHT ON THE
MOST RECENT TRIAL: large $\eta$ — the last trial or two dominate;
small $\eta$ — many past trials average together:

$$ h_T \leftarrow (1-\eta_T)\, h_T + \eta_T\, e_T, \qquad
h_D \leftarrow (1-\eta_D)\, h_D + \eta_D\, e_D. $$

**Seven learned parameters:** $\alpha_P, g_C, g_F, \beta_T, \beta_D,
\eta_T, \eta_D$.

Two honest notes. (1) A two-color display cannot separate
"enhance the target color" from "suppress the distractor color" —
only the NET effect is identified — so the color pathway carries
one signed gain (RESULTS: the g_T/g_D ridge). (2) There is no
attention window: with every item equally far from fixation, a
window would scale all six priorities by the same number, which
the gains absorb — fits with one inside, outside, and absent are
identical (tested three ways; `RESULTS.md`).


In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.2))
boxes = [(0.05, "display\n(pixels)"),
         (0.29, "priority map M(x)\n(sensory + goal + history)"),
         (0.54, "sense at item\ncenters: F_i"),
         (0.78, "softmax\nP(saccade = i)")]
for x, label in boxes:
    ax.add_patch(plt.Rectangle((x, 0.35), 0.17, 0.32, fc="#eef2fa",
                               ec="#334488", lw=1.5))
    ax.text(x + 0.085, 0.51, label, ha="center", va="center", fontsize=10)
for x, _ in boxes[:-1]:
    ax.annotate("", xy=(x + 0.24, 0.51), xytext=(x + 0.17, 0.51),
                arrowprops=dict(arrowstyle="->", lw=1.5))
ax.annotate("the goal: 'find the\ngreen circle'", xy=(0.35, 0.67),
            xytext=(0.29, 0.92), ha="center", fontsize=9,
            arrowprops=dict(arrowstyle="->", color="#118844"))
ax.annotate("previous trials' histories\n(h_T, h_D)", xy=(0.375, 0.35),
            xytext=(0.42, 0.08), ha="center", fontsize=9,
            arrowprops=dict(arrowstyle="->", color="#aa2222"))
ax.set_xlim(0, 1.0); ax.set_ylim(0, 1); ax.axis("off")
plt.title("map first; items only at the readout")
plt.show()


## 2. The displays: one example per study

Eleven experiments, pooled. Each subject searched one fixed shape
all session, so displays are reconstructed canonically: a CIRCLE
target among mixed other shapes, green items, one red color
singleton. The target never pops out — it is found by shape; the
singleton does pop out. (Each study's own color pair is shown here
for flavor; Hamblin-Frohman ships no color labels, so it falls
back to green/red.)


In [ ]:
gal = pd.read_csv("dataset/saccades_ctx.csv", low_memory=False,
                  usecols=["study", "setsize", "targCol", "singCol"])
gal = gal[gal.singCol != "none"]

fig, axes = plt.subplots(3, 4, figsize=(13, 10))
for ax, (study, sub) in zip(axes.flat, gal.groupby("study")):
    setsize = int(sub.setsize.mode()[0])
    tc, sc = sub[["targCol", "singCol"]].mode().iloc[0]
    ipos, _ = item_positions(setsize)
    tg, sg = 2, (2 + setsize // 2) % setsize + 1   # target & singleton slots
    items = [dict(x=ipos[j][0], y=ipos[j][1],
                  color=sc if (j + 1) == sg else tc,
                  shape=shape_for(j + 1, tg)) for j in range(setsize)]
    ax.imshow(render(items), origin="lower")
    ax.set_title(f"{study}: {tc}/{sc}, set size {setsize}", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
for ax in axes.flat[gal.study.nunique():]:
    ax.axis("off")
plt.suptitle("reconstructed example displays, one per study", y=0.995)
plt.tight_layout()
plt.show()



## 3. The sensory field

Before the goal can help, the display has to be sensed. The sensory
field is goal-independent: it does not know which color or shape is
the target. It only says where the rendered items create bottom-up color salience. The background is treated as
transparent: it contributes zero sensory evidence, and the color
singleton is more salient than the majority-color items.

$$ B(x) = \alpha_P\,P(x) $$

`P(x)` is fixed by the visual front end. The learned scalar
`alpha_P` decides whether that bottom-up evidence attracts or
suppresses first saccades after the goal and history fields are also
in play.


In [ ]:

# Canonical display reused through the worked example.
TARG, SING = 1, 4
pos, _ = item_positions(6)
items = [dict(x=pos[j][0], y=pos[j][1],
              color="red" if j == SING else "green",
              shape=shape_for(j + 1, TARG + 1)) for j in range(6)]
img = render(items)
maps = opponency_contrast(img)

# Placeholder gain for visualization only; Sec. 8 learns the real value.
alphaP_demo = -1.0
P_map = sensory_salience_map(items)
sensory_field = alphaP_demo * P_map

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.8))
axes[0].imshow(img, origin="lower")
axes[0].set_title("display", fontsize=10)
im1 = axes[1].imshow(P_map, origin="lower", cmap="magma", vmin=0, vmax=1)
axes[1].set_title("raw color-salience sensory evidence\nP(x)", fontsize=10)
v = np.abs(sensory_field).max()
im2 = axes[2].imshow(sensory_field, origin="lower", cmap="PuOr_r", vmin=-v, vmax=v)
axes[2].set_title("sensory field\nalpha_P * P(x)", fontsize=10)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
for ax, im in [(axes[1], im1), (axes[2], im2)]:
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
plt.tight_layout()
plt.show()



## 4. The goal field

The goal field uses the same simple form for color and shape:

$$ G(x) = g_C\,C_T(x) + g_F\,S_T(x) $$

`C_T(x)` is target-color evidence: in this two-color task, the raw
red-green contrast is signed so target-colored items are positive and
the opposite-color singleton is negative. `S_T(x)` is signed target-shape
evidence from pixel-derived template matching. The goal gains do not
create these maps; they only scale the evidence already sensed from
the display.

The gains below are placeholders; §8 learns the real ones. Each panel
is scaled to its own maximum, so compare magnitudes by the color bars.


In [ ]:

# placeholder goal gains - magnitudes picked so the demo maps are visible
# (training replaces them with fitted values)
gC_demo, gF_demo = 0.3, 14.0

u = template_axis("green")
C_T = u[0]*maps["RG"] + u[1]*maps["BY"]
# greyscale: divide by the strongest |C_T| pixel this canonical display produces
C_COLOR = np.abs(C_T).max()
C_T = C_T / C_COLOR
color_field = gC_demo * C_T

S_T = shape_match_map(render(items, scale=2))
S_T = 2 * (S_T[::2, ::2] / max(S_T.max(), 1e-9)) - 1  # greyscale: [-1, 1]
shape_field = gF_demo * S_T
goal_field = color_field + shape_field

fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))
axes[0].imshow(img, origin="lower")
axes[0].set_title("display", fontsize=10)
v = np.abs(C_T).max()
im1 = axes[1].imshow(C_T, origin="lower", cmap="PuOr_r", vmin=-v, vmax=v)
axes[1].set_title("target-color evidence\nC_T(x)", fontsize=10)
im2 = axes[2].imshow(S_T, origin="lower", cmap="PuOr_r", vmin=-1, vmax=1)
axes[2].set_title("target-shape evidence\nS_T(x)", fontsize=10)
v = np.abs(goal_field).max()
im3 = axes[3].imshow(goal_field, origin="lower", cmap="PuOr_r", vmin=-v, vmax=v)
axes[3].set_title("goal field\ng_C*C_T + g_F*S_T", fontsize=10)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
for ax, im in [(axes[1], im1), (axes[2], im2), (axes[3], im3)]:
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
plt.tight_layout()
plt.show()



In the goal field, the target is doubly bright: it has the target
color and the target shape. Plain green items carry target-color
evidence only, and the red singleton is a signed hole. Orange is
positive evidence; purple is negative evidence.

Two features of the shape map worth a second look:

- **The bright spot is smaller than the circle.** Each pixel answers
  "is a circle centered HERE?" — a template-alignment peak, not a
  filled disk. That is all the model needs: shape is read exactly at
  item centers.
- **Faint dots at the triangles.** Just off a triangle's center, the
  circle template briefly beats the triangle template, leaving thin
  rims of weak false match. Color plays no role: the image is
  binarized before shape matching.


## 5. Location-based selection history

The model remembers WHERE things have been — not what they looked
like. Two memories, one number per location, updated after every
trial as leaky accumulators:

    h = (1 - eta) * h              # everything fades a little
    h[location] += eta             # this trial's location gets a boost

`h_T` tracks target locations (recency weight `eta_T`); `h_D`
tracks distractor locations (`eta_D`). Each entry is a recency-weighted
occupancy between 0 and 1 — NOT a probability; nothing normalizes
across locations. `eta` is the weight on the newest
trial: large `eta` — recent trials dominate; small `eta` — many
past trials average together. Their map gains are
`beta_T` (past target locations PULL the eyes) and `beta_D`
(negative: past distractor locations PUSH away).

Three panels below:

1. **the accumulator** — `accumulate_history` (reused verbatim by
   the final model in §7), replayed over a ten-trial demo history:
   location 2's `h` at a large and a small `eta`;
2. **the history VECTOR** — one number per item:
   `beta_T*hT_i + beta_D*hD_i`;
3. **the history MAP** — that vector painted at the item locations
   with the fixed kernel (`selection_history_map`, sigma 0.03,
   peak 1). Added to the sensory and goal fields in §6, it completes the
   priority map.


In [ ]:
def accumulate_history(past_locs, eta):
    '''The leaky accumulator above, replayed over past trials.
    past_locs: where the item appeared on each previous trial
    (1-6; 0 = absent), oldest first. After each trial everything
    fades by (1 - eta) and that trial's location gets an eta boost.
    (A torch eta makes h a tensor, so Sec. 8's training can take
    gradients through the replay.)'''
    h = torch.zeros(6) if torch.is_tensor(eta) else np.zeros(6)
    for loc in past_locs:
        h = (1 - eta) * h
        if loc:
            h[int(loc) - 1] += eta
    return h

# the demo trial history, reused all the way to the Sec. 7 worked
# example: where things appeared on the ten previous trials,
# oldest first (1-6 = item location, 0 = absent)
past_targets    = [2, 5, 2, 2, 6, 2, 3, 2, 4, 2]   # target favored loc 2
past_singletons = [4, 4, 0, 1, 4, 4, 0, 4, 4, 4]   # singleton favored loc 4
etaT_demo, etaD_demo = 0.6, 0.2                    # demo memory speeds

# replay the SAME target history at two eta values, watch location 2
n_tr = len(past_targets)
h_fast = np.zeros((n_tr + 1, 6)); h_slow = np.zeros((n_tr + 1, 6))
for t in range(n_tr):
    h_fast[t+1] = accumulate_history(past_targets[:t+1], 0.6)
    h_slow[t+1] = accumulate_history(past_targets[:t+1], 0.2)

cx = cy = IMG / 2
px_per_unit = IMG / 1.5
yy, xx = np.mgrid[0:IMG, 0:IMG]

def selection_history_map(vals, sig=0.03):
    # each bump is the fixed Gaussian kernel with PEAK HEIGHT 1 -
    # the natural unit for a readout that SENSES the map at item
    # centers (Sec. 7): a fully primed own location senses as
    # exactly beta, and sig sets only how far the bump spreads.
    # Torch-ready: with torch vals (Sec. 7) the bump joins them as
    # a tensor so gradients reach the betas and etas
    m = 0.0
    for j in range(6):
        px, py = (pos[j][0]+0.75)/1.5*IMG, (pos[j][1]+0.75)/1.5*IMG
        bump = np.exp(-((xx-px)**2 + (yy-py)**2)
                      / (2*(sig*px_per_unit)**2))
        if torch.is_tensor(vals):
            bump = torch.as_tensor(bump)
        m = m + vals[j] * bump
    return m

# the two memories after those ten trials, built by the same
# accumulate_history the final model reuses in Sec. 7
hT_demo = accumulate_history(past_targets, etaT_demo)
hD_demo = accumulate_history(past_singletons, etaD_demo)
print("h_T:", hT_demo.round(2), " (targets, eta_T =", etaT_demo, ")")
print("h_D:", hD_demo.round(2), " (singletons, eta_D =", etaD_demo, ")")
betaT_demo, betaD_demo = 2.0, -0.5      # placeholder gains (near the
                                        # fitted values)

# ---- the history VECTOR: one number per item location ----
hist_vec = betaT_demo*hT_demo + betaD_demo*hD_demo
print("history vector:", hist_vec.round(3))

# ---- the history MAP: the vector painted with the fixed kernel ----
history_map = selection_history_map(hist_vec)

fig, axes = plt.subplots(1, 3, figsize=(14.5, 3.6))
axes[0].plot(h_fast[:, 1], color="#2233aa", label="eta = 0.6: recent trials dominate")
axes[0].plot(h_slow[:, 1], color="#aa7722", label="eta = 0.2: long average")
hit_tr = [t + 1 for t, loc in enumerate(past_targets) if loc == 2]
axes[0].scatter(hit_tr, np.full(len(hit_tr), -0.06),
                marker="|", color="#444444", label="target at location 2")
axes[0].set_xlabel("trial"); axes[0].set_ylabel("h at location 2")
axes[0].set_title("the leaky accumulator\n(replaying past_targets)", fontsize=10)
axes[0].legend(fontsize=8)
cols = ["#cc8822" if v_ >= 0 else "#553388" for v_ in hist_vec]
axes[1].bar(range(1, 7), hist_vec, color=cols)
axes[1].axhline(0, color="#999999", lw=0.8)
axes[1].set_xlabel("item"); axes[1].set_ylabel("value")
axes[1].set_title("the history vector:\n"
                  "beta_T*hT_i + beta_D*hD_i", fontsize=10)
v = np.abs(history_map).max()
im_p = axes[2].imshow(history_map, origin="lower", cmap="PuOr_r",
                      vmin=-v, vmax=v)
axes[2].plot(IMG/2, IMG/2, "k+", ms=10)
axes[2].set_title("the history map: the vector\n"
                  "painted with the fixed kernel", fontsize=10)
axes[2].set_xticks([]); axes[2].set_yticks([])
fig.colorbar(im_p, ax=axes[2], fraction=0.046, pad=0.03)
plt.tight_layout()
plt.show()



## 6. Combining sensory, goal, and history into one map

The priority map is a pixel-by-pixel sum: the sensory field (§3), the
goal field (§4), and the history map (§5). Sensory evidence, goal
evidence, and selection history compete in the same currency, in one
2-D object — and that sum is the map the readout senses (§7).

    priority(x) = sensory(x) + goal(x) + history_map(x)


In [ ]:

priority_map_demo = sensory_field + goal_field + history_map

fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
plots = [
    (sensory_field, "sensory field\nalpha_P * P(x)"),
    (goal_field, "goal field\ng_C*C_T + g_F*S_T"),
    (history_map, "history field\nselection history"),
    (priority_map_demo, "final priority map\nsensory + goal + history"),
]
for ax, (m, ttl) in zip(axes, plots):
    v = max(np.abs(m).max(), 1e-9)
    im = ax.imshow(m, origin="lower", cmap="PuOr_r", vmin=-v, vmax=v)
    ax.plot(IMG/2, IMG/2, "k+", ms=10)
    ax.set_title(ttl, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
plt.tight_layout()
plt.show()



## 7. The final model, in two small functions

Everything above, assembled — nothing new:

1. **`priority_map`** (§3-6): sensory field (`alpha_P`*P), goal
   field (`g_C`*C_T + `g_F`*S_T), plus the painted history field.
   That sum is the priority map;
2. **`predict_saccade`** (§1 steps 2-3): sense the map at the six
   item centers, softmax into P(first saccade).

All seven parameters act inside these functions: `alpha_P` scales the
sensory field, the `g` gains scale the goal evidence, the etas run
§5's `accumulate_history` on the raw past-trial locations, and the
betas weigh the resulting history field.


In [ ]:

# accumulate_history and selection_history_map are Sec. 4's - reused, not redefined

def priority_map(img, past_targets, past_singletons,
                 alpha_P, g_C, g_F, beta_T, beta_D, eta_T, eta_D,
                 target_color="green", display_items=None):
    '''The priority map M(x), from raw inputs:
    the display image plus the RAW TRIAL HISTORY - the lists of
    where the target and the singleton appeared on previous trials.

    The field decomposition mirrors the action model:
        sensory field + goal field + history field

    P is bottom-up, goal-independent color-salience evidence. C_T and S_T are
    target-color and target-shape evidence, so the goal field has the
    simple gain * evidence form: g_C*C_T + g_F*S_T.'''
    cmaps = opponency_contrast(img)
    pmap = sensory_salience_map(display_items if display_items is not None else items)
    u_ = template_axis(target_color)
    c_t = u_[0]*cmaps["RG"] + u_[1]*cmaps["BY"]              # target-color evidence
    c_t = c_t / C_COLOR                                      # greyscale: [-1, 1]
    s_t = shape_match_map(img)
    s_t = 2 * (s_t / max(s_t.max(), 1e-9)) - 1                # signed target-shape evidence

    # fixed fields become tensors HERE, where they meet learned gains
    pmap = torch.as_tensor(pmap)
    c_t = torch.as_tensor(c_t)
    s_t = torch.as_tensor(s_t)
    sensory_field = alpha_P * pmap
    goal_field = g_C * c_t + g_F * s_t

    h_T = accumulate_history(past_targets, eta_T)        # Sec. 4
    h_D = accumulate_history(past_singletons, eta_D)
    history_field = selection_history_map(beta_T*h_T + beta_D*h_D)
    return sensory_field + goal_field + torch.as_tensor(history_field)


def predict_saccade(pm):
    '''Takes the OUTPUT of priority_map() - the 2-D priority map -
    and SENSES it at each item's center (F_i = M(x_i), Sec. 1
    steps 2-3), like a range sensor reading one value per
    direction; then softmaxes the six sensed values into
    P(first saccade = i).'''
    pm = torch.as_tensor(pm)      # keeps gradients when pm carries them
    priorities = []
    for j_ in range(6):
        px_ = int(round((pos[j_][0] + 0.75) / 1.5 * IMG))
        py_ = int(round((pos[j_][1] + 0.75) / 1.5 * IMG))
        priorities.append(pm[py_, px_])
    F_ = torch.stack(priorities).float()[None]
    return torch.softmax(F_, 1), F_

print("the model: priority_map -> predict_saccade")


### One worked example, raw image to prediction

The §3-4 display fields and the §5 histories go in with placeholder
weights; out come the map, the six sensed priorities, and the
prediction. (With these placeholder gains the history bump at
item 2 dominates; §7 learns the real balance.)


In [ ]:

# the seven Sec. 1 parameters, in one place
PARAMS = dict(alpha_P=0.0, g_C=gC_demo, g_F=gF_demo,
              beta_T=betaT_demo, beta_D=betaD_demo,
              eta_T=etaT_demo, eta_D=etaD_demo)

pm = priority_map(img, past_targets, past_singletons, **PARAMS)
p_ex, F_ex = predict_saccade(pm)

print("priority per item:", F_ex[0].numpy().round(2))
print("P(first saccade): ", (p_ex[0].numpy() * 100).round(1), "%")
print("most likely saccade: item", int(F_ex[0].argmax()) + 1,
      "(the target; one PREDICTED saccade = one draw from these)")

fig, axes = plt.subplots(1, 4, figsize=(15.5, 3.4))
axes[0].imshow(img, origin="lower")
axes[0].set_title("input: the raw image\n(+ past trial locations)", fontsize=10)
pm_np = np.asarray(pm)
v = np.abs(pm_np).max()
im = axes[1].imshow(pm_np, origin="lower", cmap="PuOr_r", vmin=-v, vmax=v)
axes[1].set_title("the priority map\n(sensory + goal + history)", fontsize=10)
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.03)
for ax in axes[:2]:
    ax.set_xticks([]); ax.set_yticks([])
cols = ["#888888"] * 6
cols[TARG], cols[SING] = "#118844", "#aa2222"
axes[2].bar(range(1, 7), F_ex[0].numpy(), color=cols)
axes[2].set_title("sensed at item centers:\none priority per item", fontsize=10)
axes[2].set_xlabel("item")
axes[3].bar(range(1, 7), p_ex[0].numpy() * 100, color=cols)
axes[3].set_title("softmax -> P(first\nsaccade) per item", fontsize=10)
axes[3].set_xlabel("item"); axes[3].set_ylabel("%")
plt.tight_layout()
plt.show()


## 8. Train on the actual data

The dataset holds every first saccade from 333 people, plus the
full trial order (needed to rebuild each person's memories).
Excluded at pooling (`pool_data.py`): trials with motion/onset
singletons, and Stilwell's low-salience trials — the latter kept
entirely outside the model as the out-of-sample test in §10.

We hold out 20% of the PEOPLE and fit by maximum likelihood:
replay everyone's trials, score the probability the model gave the
real eye movement, nudge the model parameters by gradient descent.

**Why not call `priority_map()` directly in the loop?** It works —
an earlier revision did it — but one forward-and-backward through
the pixels costs ~60 ms, and 114,232 saccades x 200 epochs is ~40
days of re-rendering the SAME images: between epochs only the fitted
parameters change.

So we cache what the parameters never touch, and the point-sensing
readout makes that cache tiny: per display, the three sensed fields'
values at six pixels, plus the history kernel between the centers.
`priority_map` builds its own cache — the map is linear in the
gains. Each epoch reassembles the
sensed values with the current parameters. Nothing is approximated; the next cell proves it on a trial.


In [ ]:

# build the exact cache: the Sec. 7 model, sensed at the item centers
import data

sacc = pd.read_csv("dataset/saccades_ctx.csv", low_memory=False)  # a row per first saccade
ev = pd.read_csv("dataset/events.csv", low_memory=False)          # a row per trial
sacc, tt = data.build_tensors(sacc, ev)   # a few minutes: aligns everything
N = len(sacc)

def items_for(setsize, targ, sing, tc="green", sc="red"):
    ipos, _ = item_positions(setsize)
    return [dict(x=ipos[j][0], y=ipos[j][1],
                 color=(sc if j + 1 == sing else tc),
                 shape=shape_for(j + 1, targ)) for j in range(setsize)]

def display_for(setsize, targ, sing, tc="green", sc="red"):
    '''Re-render a display from its trial parameters (Sec. 2/3).'''
    return render(items_for(setsize, targ, sing, tc, sc))

def channel_maps(img_, items_, tc="green"):
    # Model-facing sensed fields: P, C_T, S_T.
    cmaps = opponency_contrast(img_)
    pmap = sensory_salience_map(items_)
    u_ = template_axis(tc)
    c_t = (u_[0]*cmaps["RG"] + u_[1]*cmaps["BY"]) / C_COLOR
    s_t = shape_match_map(img_)
    s_t = 2 * (s_t / max(s_t.max(), 1e-9)) - 1
    return torch.stack([torch.as_tensor(pmap),
                        torch.as_tensor(c_t),
                        torch.as_tensor(s_t)])

def centers(setsize):
    # the sensed pixels: each item's center (Sec. 7's predict_saccade)
    ipos, _ = item_positions(setsize)
    return [(int(round((p[1]+0.75)/1.5*IMG)), int(round((p[0]+0.75)/1.5*IMG)))
            for p in ipos]

# every display any saccade saw, sensed: A_CH[display, item, field]
keys = sorted(set(zip(sacc.setsize.astype(int),
                      sacc.targLoc.astype(int), sacc.singLoc.astype(int))))
A_CH = torch.zeros(len(keys), 6, 3)
for d_, key in enumerate(keys):
    items_ = items_for(*key)
    ch = channel_maps(display_for(*key), items_)
    for j_, (py_, px_) in enumerate(centers(key[0])):
        A_CH[d_, j_] = ch[:, py_, px_].float()
# no extra unit step: the fields are greyscale (Sec. 3), so the
# cached values already ARE the model's units
DIDX = {key: i for i, key in enumerate(keys)}
didx = torch.tensor([DIDX[key] for key in
                     zip(sacc.setsize.astype(int), sacc.targLoc.astype(int),
                         sacc.singLoc.astype(int))])

# the Sec. 4 history kernel sensed between item centers (peak 1 at
# its own center, ~0 at the neighbors), per set size
B_HIST = {}
for s_ in (6, 4):
    ipos, _ = item_positions(s_)
    cs = centers(s_)
    Bm = np.zeros((6, 6))
    for j_ in range(s_):
        pxj = (ipos[j_][0]+0.75)/1.5*IMG; pyj = (ipos[j_][1]+0.75)/1.5*IMG
        bump = np.exp(-((xx-pxj)**2 + (yy-pyj)**2)
                      / (2*(0.03*px_per_unit)**2))
        for i_ in range(s_):
            Bm[i_, j_] = bump[cs[i_][0], cs[i_][1]]
    B_HIST[s_] = torch.tensor(Bm)
m6 = torch.tensor((sacc.setsize == 6).values)

n_subj = tt["eT"].shape[0]
rng = np.random.default_rng(0)
test_subj = torch.zeros(n_subj, dtype=torch.bool)
test_subj[rng.choice(n_subj, n_subj // 5, replace=False)] = True
test = test_subj[tt["si"]]; train = ~test
print(f"{N} saccades, {len(keys)} displays sensed "
      f"({A_CH.numel()} cached numbers); "
      f"{int((~test_subj).sum())} people to train on, "
      f"{int(test_subj.sum())} held out")


**Sanity check.** One trial through both routes with the same
parameters: raw pixels through `priority_map()` ->
`predict_saccade()`, and the cached reassembly. The six priorities
must agree to float precision — then training on the cache IS
training the §7 model.


In [ ]:

def cached_F(key, h_T, h_D, alpha_P, g_C, g_F, beta_T, beta_D):
    # reassemble the six SENSED priorities from cached pieces
    s_ = key[0]
    items_ = items_for(*key)
    ch = channel_maps(display_for(*key), items_)
    vals = torch.as_tensor(beta_T*h_T + beta_D*h_D)  # literal model exactly
    F_ = []
    for i_, (py_, px_) in enumerate(centers(s_)):
        stim = alpha_P*ch[0, py_, px_] + g_C*ch[1, py_, px_] + g_F*ch[2, py_, px_]
        hist = (vals[:s_] * B_HIST[s_][i_, :s_]).sum()
        F_.append(stim + hist)
    return torch.stack(F_)

pt, ps = past_targets, past_singletons          # the Sec. 5 demo history
key0 = (6, 1, 4)                      # target slot 1, singleton slot 4
F_lit = predict_saccade(priority_map(display_for(*key0), pt, ps, **PARAMS,
                                      display_items=items_for(*key0)))[1][0]
F_c = cached_F(key0,
               accumulate_history(pt, PARAMS["eta_T"]),
               accumulate_history(ps, PARAMS["eta_D"]),
               PARAMS["alpha_P"], PARAMS["g_C"], PARAMS["g_F"],
               PARAMS["beta_T"], PARAMS["beta_D"])
print("literal:", F_lit.numpy().round(6))
print("cached: ", F_c.float().numpy().round(6))
print(f"max |difference|: {float((F_lit - F_c.float()).abs().max()):.2e}")


### The fit, at scale

Per epoch: replay the memories (the etas are learning) and
reassemble every saccade's six priorities from the cached values —
under a second per epoch for all 114,232 saccades. Set-size-4
displays (Hamblin 2022) sense four centers. No unit bookkeeping:
the sensed fields are greyscale (§3-4), so the cached values already are
the model's units.


In [ ]:

def build_memories(eta_T, eta_D):
    # replay every subject's trials in order; memory AS OF each trial
    S_, T_, _ = tt["eT"].shape
    hT = torch.zeros(S_, 6); hD = torch.zeros(S_, 6)
    outT, outD = [], []
    for t in range(T_):
        outT.append(hT); outD.append(hD)
        hT = (1 - eta_T) * hT + eta_T * tt["eT"][:, t]
        hD = (1 - eta_D) * hD + eta_D * tt["eD"][:, t]
    return torch.stack(outT, 1), torch.stack(outD, 1)

def model_F(w, raw_eta, A_=None):
    """Every saccade's six sensed priorities: the Sec. 7 model
    reassembled from the cache. Between epochs only the memories
    are recomputed; the gains just reweigh cached numbers. A_
    substitutes alternative sensed fields for the same trials
    (Sec. 10b's counterfactual displays); default: the real ones."""
    eta_T = torch.sigmoid(raw_eta["T"]); eta_D = torch.sigmoid(raw_eta["D"])
    memT, memD = build_memories(eta_T, eta_D)
    hT = memT[tt["si"], tt["ti"]]; hD = memD[tt["si"], tt["ti"]]
    vals = w["beta_T"] * hT + w["beta_D"] * hD
    Ad = (A_CH if A_ is None else A_)[didx]        # sensed fields: P, C_T, S_T
    stimulus = w["alpha_P"]*Ad[..., 0] + w["g_C"]*Ad[..., 1] + w["g_F"]*Ad[..., 2]
    F = torch.zeros(N, 6)
    for s_ in (6, 4):
        msk = m6 if s_ == 6 else ~m6
        F[msk] = stimulus[msk] + vals[msk] @ B_HIST[s_].float().T
    return F, memT, memD

w = {name: torch.tensor(v, requires_grad=True) for name, v in
     [("alpha_P", 0.0), ("g_C", 0.5), ("g_F", 1.0),
      ("beta_T", 0.5), ("beta_D", -0.1)]}
raw_eta = {n: torch.tensor(0.0, requires_grad=True) for n in ["T", "D"]}
optimizer = torch.optim.Adam(list(w.values()) + list(raw_eta.values()), lr=0.05)

losses = []
for epoch in range(200):
    F, _, _ = model_F(w, raw_eta)
    F = F.masked_fill(~tt["valid"], -1e9)
    logp = torch.log_softmax(F, 1).gather(
        1, tt["choice"][:, None]).squeeze(1)
    loss = -logp[train].mean()
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    losses.append(loss.item())
    if epoch % 25 == 0:
        print(f"epoch {epoch}: loss {loss.item():.4f}", flush=True)

plt.figure(figsize=(6, 3))
plt.plot(losses)
plt.xlabel("training step"); plt.ylabel("loss (per saccade)")
plt.title("training on the real eye movements")
plt.show()


In [ ]:

learned = {k: v.item() for k, v in w.items()}
learned["eta_T"] = torch.sigmoid(raw_eta["T"]).item()
learned["eta_D"] = torch.sigmoid(raw_eta["D"]).item()

for name, meaning in [
        ("alpha_P", "bottom-up sensory color salience"),
        ("g_C", "up-weights target color, down-weights distractor color"),
        ("g_F", "up-weights the target shape"),
        ("beta_T", "pull toward past target locations"),
        ("beta_D", "push from past distractor locations"),
        ("eta_T", "weight on recent target locations"),
        ("eta_D", "weight on recent distractor locations")]:
    print(f"{name:8} = {learned[name]:+7.3f}   {meaning}")


## 9. Evaluation: how good is it? (held-out people only)

Scored ONLY on people the model never saw. Three measures:

- **Mean probability on the true choice** — the geometric-mean
  probability the model assigned to the item actually fixated.
  Chance: 1/(items on screen), ~18% here.
- **Top-1 accuracy** — how often the actual choice was the model's
  highest-probability item. Chance ~18%.
- **Pseudo-R-squared** — `1 - NLL_model / NLL_chance`. 0 = no
  better than guessing; 1 = perfect. For human choice data,
  0.2-0.4 counts as excellent — the same person, display, and
  history does not always yield the same saccade, so 1.0 is
  unreachable in principle.


In [ ]:

with torch.no_grad():
    F, memT, memD = model_F(w, raw_eta)
    F = F.masked_fill(~tt["valid"], -1e9)
    prob = torch.softmax(F, 1)
    p_true = prob.gather(1, tt["choice"][:, None]).squeeze(1)

nll = -p_true[test].log().mean()
chance = torch.log(tt["valid"].sum(1).float())[test].mean()
print(f"held-out saccades: {int(test.sum())}")
print(f"mean probability on the true choice: "
      f"{p_true[test].log().mean().exp()*100:.1f}%  "
      f"(chance {torch.exp(-chance)*100:.1f}%)")
print(f"top-1 accuracy: "
      f"{(prob.argmax(1) == tt['choice'])[test].float().mean()*100:.1f}%")
print(f"pseudo-R-squared vs chance: {1 - nll/chance:.3f}")
print(f"held-out NLL per saccade: {nll:.5f}")


**Interpretation.** The model roughly halves a stranger's
first-saccade uncertainty: ~25% geometric-mean probability on the
true choice vs ~18% chance, the right item as top guess almost
half the time, pseudo-R-squared ~0.2. For calibration:
memorization benchmarks built from the training people (choice
frequencies per display, even per person) score WORSE — the graded
history traces carry information no frequency table can (RESULTS,
"Oracle benchmarks").


## 10. Does it behave like people? Three signature patterns

Held-out people only; nothing below was fitted to these patterns —
they fall out of the one trained equation.

**9a. Oculomotor suppression.** First saccades go to the salient
red singleton LESS often than to an average plain item — the
below-baseline suppression this dataset exists to measure.


In [ ]:
first = torch.tensor((sacc.saccindex == 1).values)  # all True: the scope
sing_present = torch.tensor((sacc.singLoc > 0).values)
rows = torch.arange(N)
isT = torch.zeros(N, 6); isT[rows, torch.tensor(sacc.targLoc.values) - 1] = 1
isS = torch.zeros(N, 6)
sp = torch.tensor(sacc.singLoc.values)
isS[rows[sp > 0], sp[sp > 0] - 1] = 1
chose = torch.zeros(N, 6); chose[rows, tt["choice"]] = 1

m = test & first & sing_present

# REGENERATE the test saccades: one model saccade per real trial,
# drawn from the predicted probabilities (each trial conditioned on
# the subject's actual history). The model line then carries the
# same trial-count binomial noise as the people line, so its error
# bars and t-test are commensurable with the data's.
gen = torch.Generator().manual_seed(0)
sim = torch.multinomial(prob[m], 1, generator=gen).squeeze(1)
simch = torch.zeros(int(m.sum()), 6)
simch[torch.arange(int(m.sum())), sim] = 1

def subject_rates(ch_):
    """One set of saccades (0/1 choice rows over mask m) -> each
    held-out subject's landing rates (%): columns t (target),
    ns (plain-item average), s (singleton)."""
    d_ = pd.DataFrame(dict(
        si=tt["si"][m].numpy(),
        t=(ch_ * isT[m]).sum(1).numpy(),
        s=(ch_ * isS[m]).sum(1).numpy(),
        n_pl=(tt["valid"][m].sum(1).float() - 2).clamp(min=1).numpy()))
    d_["ns"] = (1 - d_.t - d_.s) / d_.n_pl
    return d_.drop(columns="n_pl").groupby("si").mean() * 100

def rates_long(frames_, cond_name):
    """{condition: subject_rates frame} -> long form for seaborn:
    one row per subject x condition x item type."""
    cat = pd.concat([fr[["t", "ns", "s"]].reset_index().assign(**{cond_name: k})
                     for k, fr in frames_.items()])
    return cat.melt(id_vars=["si", cond_name], var_name="item",
                    value_name="rate")

frames = {"human": subject_rates(chose[m]), "model": subject_rates(simch)}

# the suppression effect (non-singleton minus singleton), tested
# separately per line: paired t-test across subjects
import pingouin as pg
tests = {src_: pg.ttest(fr["ns"], fr["s"], paired=True)
         for src_, fr in frames.items()}

def p_str(t_):
    p = t_["p_val"].iloc[0]
    return "p < .001" if p < .001 else f"p = {p:.3f}"

# seaborn does the means and the bootstrapped 95% CIs (over the
# subject-level rows) itself
import seaborn as sns
plt.figure(figsize=(5.4, 4.4))
ax = sns.pointplot(data=rates_long(frames, "source"), x="item", y="rate",
                   hue="source", order=["t", "ns", "s"],
                   hue_order=["human", "model"],
                   #palette={"human": "#444444", "model": "#2233aa"},
                   errorbar=("ci", 95), n_boot=10000, seed=0, capsize=0.03)
supp = {src_: fr["ns"].mean() - fr["s"].mean()
        for src_, fr in frames.items()}
# the suppression bracket: non-singleton vs singleton distractor
top = max(fr["ns"].mean() for fr in frames.values()) + 4
ax.plot([1, 1, 2, 2], [top - 1.2, top, top, top - 1.2], "k-", lw=1)
ax.text(1.5, 30, "Oculomotor\nSuppression Effect", ha="center",
        fontsize=10, fontweight="bold")
ax.text(1.5, 25, f"human {supp['human']:.1f}%",
        ha="center", fontsize=10)
ax.text(1.5, 22, f"model {supp['model']:.1f}%",
        ha="center", fontsize=10)
ax.set_xticks([0, 1, 2], ["Target", "Non-singleton\nDistractor",
                          "Singleton\nDistractor"])
ax.set_xlabel("") 
ax.set_ylabel("First Eye Movements (%)", fontsize=15)
ax.set_ylim(0, 50)
ax.legend(loc="lower left", frameon=False, title=None)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

for src_ in ("human", "model"):
    print(f"{src_}: non-singleton vs singleton distractor "
          f"(paired across {len(frames[src_])} held-out subjects)")
    print(tests[src_].round(4).to_string(index=False), "\n")


**9b. High- vs low-salience suppression (Stilwell).** Stilwell
(2023) found MORE suppression for MORE salient singletons. A
strong test: the low-salience trials were excluded from our data
entirely (§8) — the model has never seen them. We apply the
trained weights, unchanged, to displays rendered in the pipeline's
schematic colors (a red vs a less-red family; the claim is the
qualitative gradient, not Stilwell's exact colors or numbers).

(Design: the real held-out trials ARE the high-salience
condition — green target, red singleton. We run one counterfactual
second pass over the SAME trials with the singleton re-rendered in
teal (near the target color = low salience), keeping each
subject's real selection history, and regenerate saccades from
both passes. One line per condition, all three item types; the
suppression difference is the salience x item-type interaction in
a repeated-measures ANOVA over the held-out subjects. For
reference, people show the same gradient: 7.0% vs 11.3% first
saccades to the singleton — from the full battery over all of
Stilwell's color pairs, `stilwell_salience.py`; the raw file is
not in the repo.)


In [ ]:
# The REAL held-out trials are the HIGH-salience condition: every
# display is a green target with a RED singleton (opposite color).
# Counterfactual second pass, Han's design: re-sense the SAME
# trials with a TEAL singleton (near the target color = LOW
# salience), keep each subject's real selection history, and
# regenerate saccades for both passes. High pass: 9a's simulated
# saccades; low pass: drawn below.
A_LOW = torch.zeros_like(A_CH)
for d_, key in enumerate(keys):
    items_ = items_for(*key, sc="teal")
    ch = channel_maps(display_for(*key, sc="teal"), items_)
    for j_, (py_, px_) in enumerate(centers(key[0])):
        A_LOW[d_, j_] = ch[:, py_, px_].float()

with torch.no_grad():
    F_low, _, _ = model_F(w, raw_eta, A_LOW)
prob_low = torch.softmax(F_low.masked_fill(~tt["valid"], -1e9), 1)
gen3 = torch.Generator().manual_seed(2)
sim_low = torch.multinomial(prob_low[m], 1, generator=gen3).squeeze(1)
simch_low = torch.zeros(int(m.sum()), 6)
simch_low[torch.arange(int(m.sum())), sim_low] = 1

# the high pass IS 9a: same saccades, same per-subject frame -
# only the low pass is new
passes = {"High salience": frames["model"],
          "Low salience": subject_rates(simch_low)}
long9b = rates_long(passes, "salience")

# suppression difference across salience = the salience x item-type
# interaction: 2 (high/low) x 2 (non-singleton/singleton)
# repeated-measures ANOVA across the held-out subjects
aov = pg.rm_anova(dv="rate", within=["salience", "item"], subject="si",
                  data=long9b[long9b["item"].isin(["ns", "s"])],
                  detailed=True)

plt.figure(figsize=(5.4, 4.4))
ax = sns.pointplot(data=long9b, x="item", y="rate", hue="salience",
                   order=["t", "ns", "s"], hue_order=list(passes),
                   palette={"High salience": "#aa2222",
                            "Low salience": "#88abdd"},
                   errorbar=("ci", 95), n_boot=10000, seed=0, capsize=0.03)
supp_ = {sal: p_["ns"].mean() - p_["s"].mean()
         for sal, p_ in passes.items()}
# the suppression bracket: non-singleton vs singleton distractor
top = max(p_["ns"].mean() for p_ in passes.values()) + 5
ax.plot([1, 1, 2, 2], [top - 1.2, top, top, top - 1.2], "k-", lw=1)
ax.text(1.5, 30, "Oculomotor\nSuppression Effect", ha="center",
        fontsize=10, fontweight="bold")
ax.text(1.5, 25, f"high salience: {supp_['High salience']:.1f}%",
        ha="center", fontsize=10)
ax.text(1.5, 22, f"low salience: {supp_['Low salience']:.1f}%",
        ha="center", fontsize=10)
ax.set_xticks([0, 1, 2], ["Target", "Non-singleton\nDistractor",
                          "Singleton\nDistractor"])
ax.set_xlabel(""); ax.set_ylabel("First Eye Movements (%)", fontsize=15)
ax.set_ylim(0, 50)
ax.legend(loc="lower left", frameon=False, title=None)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

print("condition means over held-out subjects (%):")
print(pd.DataFrame({sal: p_[["t", "ns", "s"]].mean()
                    for sal, p_ in passes.items()}).T.round(1).to_string(),
      "\n")
print(f"2 x 2 repeated-measures ANOVA (unit: "
      f"{long9b.si.nunique()} held-out subjects)")
print(aov.round(4).to_string(index=False))


The mechanism: `g_C` weighs each item's projection onto the
goal's color axis. A low-salience singleton sits NEAR the target
color, so its projection is only mildly negative — it largely
escapes suppression. A high-salience singleton points the opposite
way and takes the full hit. The gradient falls out of one signed
gain applied to colors the fit never saw.

**9c. Target location priming.** Saccades to the target roughly
double when the target repeats its location — the
selection-history machinery at work.


In [ ]:
ev2 = ev.copy()
ev2["block"] = pd.to_numeric(ev2["block"], errors="coerce").fillna(0.0)
ev2["trial"] = pd.to_numeric(ev2["trial"], errors="coerce")
ev2["subj"] = ev2["subj"].astype(str)
ev2 = ev2.sort_values(["study", "subj", "block", "trial"])
ev2["prevT"] = ev2.groupby(["study", "subj"]).targLoc.shift(1)
key = ["study", "subj", "block", "trial"]
sacc2 = sacc.merge(ev2[key + ["prevT"]], on=key, how="left")

# regenerated saccades for ALL trials (9a's logic, whole dataset):
# one draw per real trial from the predicted probabilities
gen2 = torch.Generator().manual_seed(1)
sim2 = torch.multinomial(prob, 1, generator=gen2).squeeze(1)
simch2 = torch.zeros(N, 6); simch2[torch.arange(N), sim2] = 1

def priming_panel(ax, m_change, m_repeat, pick, effect_name, what):
    """Paper-style priming panel: Change vs Repeat bars (subjects +
    model from the regenerated saccades), per-subject means over
    held-out subjects; seaborn computes the bootstrapped 95% CIs.
    Bracket: effect (repeat - change), p, and d_z from a paired
    t-test across subjects."""
    fr_ = {}
    for cond, m_ in (("Change", m_change), ("Repeat", m_repeat)):
        d_ = pd.DataFrame(dict(
            si=tt["si"][m_].numpy(),
            human=(chose[m_] * pick[m_]).sum(1).numpy(),
            model=(simch2[m_] * pick[m_]).sum(1).numpy()))
        fr_[cond] = d_.groupby("si").mean() * 100
    both = fr_["Change"].join(fr_["Repeat"], lsuffix="_ch",
                              rsuffix="_rep").dropna()
    tests_ = {src_: pg.ttest(both[src_ + "_rep"], both[src_ + "_ch"],
                             paired=True) for src_ in ("human", "model")}
    long_ = pd.concat([fr.loc[both.index].reset_index().assign(cond=cond)
                       for cond, fr in fr_.items()])
    long_ = long_.melt(id_vars=["si", "cond"], var_name="source",
                       value_name="rate")
    sns.barplot(data=long_, x="cond", y="rate", hue="source",
                order=["Change", "Repeat"],
                hue_order=["human", "model"],
                palette={"human": "#AE3B5F", "model": "#aa9322"},
                errorbar=("ci", 95), n_boot=10000, seed=0,
                capsize=0.08, ax=ax)
    top = long_.groupby(["cond", "source"]).rate.mean().max() * 1.08
    ax.plot([0, 0, 1, 1], [top, top * 1.04, top * 1.04, top],
            "k-", lw=1)
    ax.text(0.5, top * 1.36, effect_name, ha="center", fontsize=11,
            fontweight="bold")
    for k, src_ in enumerate(("human", "model")):
        eff = both[src_ + "_rep"].mean() - both[src_ + "_ch"].mean()
        ax.text(0.5, top * (1.26 - 0.10 * k),
                f"{src_} {eff:.1f}%",
                ha="center", fontsize=10)
    ax.set_xlabel(what, fontsize=12, fontweight="bold")
    ax.set_ylabel(f"First Eye Movements\nto {what.split()[0]} (%)", fontsize=15)
    ax.set_ylim(0, top * 1.46)
    return tests_, len(both)

fig, ax = plt.subplots(figsize=(5.4, 4.4))
tr = test & first & torch.tensor(sacc2.prevT.values == sacc.targLoc.values)
tc_ = test & first & torch.tensor((sacc2.prevT.values != sacc.targLoc.values)
                                  & ~np.isnan(sacc2.prevT.values))
tests_T, nT = priming_panel(ax, tc_, tr, isT, "Target Location Priming",
                            "Target Location")
ax.legend(loc="center left", frameon=False, title=None)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

for src_ in ("human", "model"):
    print(f"{src_}, target location: repeat vs change "
          f"(paired across {nT} held-out subjects)")
    print(tests_T[src_].round(4).to_string(index=False), "\n")


## 11. The spatial prior, trial by trial

Section 5 used made-up history; here are the REAL trained memories
of one held-out person. Before each display appears, the prior
`beta_T*h_T + beta_D*h_D` already tilts toward their last target
location — the green circle marks it; watch the bump follow it
from trial to trial. The right bar chart quantifies this across
all held-out trials.


In [ ]:
si_np = tt["si"].numpy(); ti_np = tt["ti"].numpy()
subj = int(np.unique(si_np[test.numpy()])[0])       # one held-out person
trials = np.sort(ti_np[(si_np == subj)])[40:44]     # four consecutive trials

with torch.no_grad():
    fig, axes = plt.subplots(1, 5, figsize=(17, 3.4))
    for ax, t in zip(axes[:4], trials):
        hT_t = memT[subj, t].numpy(); hD_t = memD[subj, t].numpy()
        # the prior, location by location (learned weights)...
        prior_t = (learned["beta_T"]*hT_t
                   + learned["beta_D"]*hD_t)
        pm = selection_history_map(prior_t)     # ...then painted, display only
        v = max(np.abs(pm).max(), 1e-9)
        ax.imshow(pm, origin="lower", cmap="PuOr_r", vmin=-v, vmax=v)
        prev = int(tt["eT"][subj, t-1].argmax()) if t > 0 else None
        if prev is not None:
            px, py = (pos[prev][0]+0.75)/1.5*IMG, (pos[prev][1]+0.75)/1.5*IMG
            ax.add_patch(plt.Circle((px, py), 14, fill=False,
                                    ec="#118844", lw=2.2))
        ax.set_title(f"prior before trial {int(t)}", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])

    # across all held-out trials: prior at prev-target loc vs elsewhere
    prior_slots = learned["beta_T"]*memT + learned["beta_D"]*memD
    at_prev, at_other = [], []
    for s in np.unique(si_np[test.numpy()]):
        eT_s = tt["eT"][int(s)]
        T_ = int(ti_np[si_np == int(s)].max()) + 1
        for t in range(1, T_):
            prev = int(eT_s[t-1].argmax())
            if eT_s[t-1].sum() == 0:
                continue
            vals = prior_slots[int(s), t].numpy()
            at_prev.append(vals[prev])
            at_other.append(np.delete(vals, prev).mean())
    axes[4].bar([0, 1], [np.mean(at_prev), np.mean(at_other)],
                color=["#118844", "#888888"])
    axes[4].set_xticks([0, 1])
    axes[4].set_xticklabels(["previous\ntarget loc", "average\nother loc"],
                            fontsize=9)
    axes[4].set_ylabel("mean spatial prior")
    axes[4].set_title("all held-out trials", fontsize=10)
    plt.tight_layout()
    plt.show()
print(f"prior at previous target location: {np.mean(at_prev):+.3f}  "
      f"vs other locations: {np.mean(at_other):+.3f}")



## Recap

| The model receives | It returns |
| --- | --- |
| a picture, a goal, the trial history | a probability per item |

We laid out the structure (§1) and displays (§2), introduced the
sensory field (§3), built the target-color and target-shape goal
field (§4), built the leaky-accumulator history map (§5), summed
sensory + goal + history into the priority map (§6), assembled
`priority_map` -> `predict_saccade` with a worked example (§7),
trained the seven parameters on all the real first saccades through
an exact cache (§8), evaluated on people the model never saw (§9),
regenerated saccades for the held-out trials and reproduced three
signatures with subject-level statistics — oculomotor suppression,
the salience gradient under a counterfactual low-salience singleton,
and target-location priming (§10) — and watched the spatial prior
track the previous target trial by trial (§11). (Why no attention
window? See the §1 note: in this scope it is a shared scalar the
gains absorb — tested three ways in `RESULTS.md`.)

More: `RESULTS.md` (the results ledger) and
`docs/priority_field_visual_search_model.md` (the theory).
